## PropertyLens RAG — Notebook C: Inference + hybrid price (v4.1+)

**Purpose:** Same query-time pipeline as Notebook B ([`05_propertylens_rag_inference (1).ipynb`](05_propertylens_rag_inference%20(1).ipynb)) — Pinecone hybrid retrieval, multi-query, weighted RRF, cross-encoder, MMR — plus **optional** hybrid-cluster resale price prediction via [`yc_hybrid_inference.predict_from_user_input`](../../03_ml_layer_hybrid/yc_hybrid_inference.py).

**Assumes:** Notebook A (`04_propertylens_build_index.ipynb`) has already run. This notebook does not build the index.

**vs Notebook B:** Loading `hybrid_cluster_bundle.joblib` adds roughly **~1 GB RSS** on top of BGE-M3 + cross-encoder + Ollama. Prediction runs only when `ENABLE_PRICE_PREDICTION` is on and the query looks like a price-estimate request with enough structured detail (or use env `ENABLE_PRICE_PREDICTION=false` to match Notebook B memory profile).

**Before running:** After a kernel crash or if memory is tight, **restart the kernel** and run from the top. The memory cell defines `cleanup_memory()` — it is invoked automatically before dense encoder, cross-encoder, and hybrid bundle loads.

**Feature table:** Full POI-backed rows need a recent `hdb_feature_table_*.csv` on disk (see `yc_hybrid_inference.default_feature_table_csv()`), or pass `YC_CSV_PATH` / `PROPERTYLENS_YC_CSV`.

**Memory discipline:** Cross-encoder on CPU; `mem()` checkpoints; hybrid bundle loaded once when enabled.

![PropertyLens RAG inference pipeline](../../images/Screenshot%202026-04-17%20at%201.29.11%E2%80%AFPM.png)


### Install dependencies

In [1]:
# Inference deps + joblib for hybrid_cluster_bundle (optional load below).
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama pandas numpy python-dotenv psutil joblib


### Configuration

In [2]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone (must match Notebook A) ─────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY in repo-root .env"

# ── Models ────────────────────────────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Retrieval knobs ───────────────────────────────────────────────────────────
TOP_K_RETRIEVAL = 50
TOP_K_RERANK    = 10
TOP_K_MMR       = 5
TOP_K_FINAL     = 5
MMR_LAMBDA      = 0.7
RRF_K           = 60
N_SUBQUERIES    = 3

# Source weights for weighted RRF — boosts amenity/xai chunks against the
# much-larger transactions pool.
SOURCE_WEIGHTS: dict[str, float] = {
    "transaction": 1.0,
    "amenity":     2.5,
    "trend":       1.0,
    "xai":         2.5,
}

# ── LLM ───────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"

# ── Device pinning ────────────────────────────────────────────────────────────
# Pin cross-encoder to CPU. On Mac with Ollama running, MPS + Gemma + bi-encoder
# + cross-encoder compete for the same memory pool; CPU for CE is the single
# most effective stability fix.
CROSS_ENCODER_DEVICE = "cpu"

# ── Paths (must match Notebook A) ────────────────────────────────────────────
def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root.")

REPO_ROOT       = find_repo_root()

# ── Hybrid price model (Notebook C only) ───────────────────────────────────
ENABLE_PRICE_PREDICTION = os.getenv("ENABLE_PRICE_PREDICTION", "true").lower() in ("1", "true", "yes")
_hbp = os.getenv("HYBRID_BUNDLE_PATH") or os.getenv("PROPERTYLENS_HYBRID_BUNDLE")
HYBRID_BUNDLE_PATH = Path(_hbp) if _hbp else None
_yc = os.getenv("YC_CSV_PATH") or os.getenv("PROPERTYLENS_YC_CSV")
YC_CSV_PATH = Path(_yc) if _yc else None

BM25_CACHE_PATH = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"

print("Config loaded.")
print(f"  Pinecone index     : {PINECONE_INDEX}")
print(f"  BM25 cache path    : {BM25_CACHE_PATH}")
print(f"  CE device          : {CROSS_ENCODER_DEVICE}")
print(f"  Source weights     : {SOURCE_WEIGHTS}")
print(f"  Price prediction   : {ENABLE_PRICE_PREDICTION}")
print(f"  Hybrid bundle path : {HYBRID_BUNDLE_PATH or '(default)'}")
print(f"  YC CSV override    : {YC_CSV_PATH or '(auto)'}")


Config loaded.
  Pinecone index     : propertylens-rag
  BM25 cache path    : /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  CE device          : cpu
  Source weights     : {'transaction': 1.0, 'amenity': 2.5, 'trend': 1.0, 'xai': 2.5}
  Price prediction   : True
  Hybrid bundle path : (default)
  YC CSV override    : (auto)


### Memory helpers

`cleanup_memory()` runs GC twice and clears CUDA/MPS allocator caches when PyTorch is loaded — **call it before re-running heavy cells** after a crash or when RSS is high.

`mem(label)` runs `cleanup_memory()` then prints RSS. Use it before `hybrid_cluster_bundle.joblib` loads if you see kernel OOM.

**If the kernel dies:** restart the kernel, run cells from the top, and set env `ENABLE_PRICE_PREDICTION=false` to skip the ~1 GB joblib bundle.


In [3]:
from __future__ import annotations
import gc
import psutil

_PROC = psutil.Process(os.getpid())

def rss_mb() -> float:
    return _PROC.memory_info().rss / (1024 * 1024)


def cleanup_memory() -> None:
    """Aggressive GC + PyTorch allocator release. Call before heavy loads (joblib, transformers)."""
    gc.collect()
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception:
        pass


def mem(label: str) -> None:
    cleanup_memory()
    print(f"  [MEM] {label:<32s} RSS = {rss_mb():8.1f} MB")


cleanup_memory()
mem("startup")


  [MEM] startup                          RSS =    248.6 MB


### Connect to Pinecone

No `create_index` — Notebook A should have populated it already. If the index is empty, retrieval will return zero chunks.


In [4]:
from __future__ import annotations
from pinecone import Pinecone

pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(stats)
total = stats.get("total_vector_count") if isinstance(stats, dict) else getattr(stats, "total_vector_count", 0)
if not total:
    print("\n⚠ Pinecone index appears empty. Run Notebook A first.")


/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 05:46:47 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '6',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '17'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 1995},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 204}},
 'storageFullness': 0.0,
 'total_vector_count': 2596,
 'vector_type': 'dense'}


### Load encoders

Dense encoder (BGE-M3) on default device. BM25 loaded from cache — **fails loudly if the cache is missing** rather than silently refitting a different encoder than the one used at upsert.


In [5]:
from __future__ import annotations
import pickle
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


cleanup_memory()
mem("before dense encoder + BM25 load")


def load_bm25_from_cache(cache_path: Path) -> BM25Encoder:
    if not cache_path.exists():
        raise FileNotFoundError(
            f"BM25 cache not found: {cache_path}\n"
            f"Run Notebook A (04_propertylens_build_index.ipynb) first."
        )
    with cache_path.open("rb") as f:
        return pickle.load(f)


dense_encoder = SentenceTransformer(DENSE_MODEL_NAME)
bm25_encoder  = load_bm25_from_cache(BM25_CACHE_PATH)
print(f"Dense encoder : {DENSE_MODEL_NAME}")
print(f"BM25 encoder  : loaded from {BM25_CACHE_PATH}")
mem("after encoders loaded")


  [MEM] before dense encoder + BM25 load RSS =    508.8 MB


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39865.16it/s]


Dense encoder : BAAI/bge-m3
BM25 encoder  : loaded from /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  [MEM] after encoders loaded            RSS =    993.2 MB


### Load cross-encoder (pinned to CPU)

Explicitly on CPU to sidestep MPS/CUDA contention with Ollama.


In [ ]:
from __future__ import annotations
from typing import Any, Tuple
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


cleanup_memory()
mem("before cross-encoder load")


def load_cross_encoder(model_name: str, device: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model, pinned to device."""
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tok, model


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL, CROSS_ENCODER_DEVICE)
print(f"Cross-encoder loaded on device: {CROSS_ENCODER_DEVICE}")
mem("after cross-encoder loaded")


  [MEM] before cross-encoder load        RSS =    993.2 MB


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 17211.13it/s]


Cross-encoder loaded on device: cpu
  [MEM] after cross-encoder loaded       RSS =   1313.8 MB


: 

### Hybrid cluster bundle (optional)

Loads `hybrid_cluster_bundle.joblib` from [`03_ml_layer_hybrid/artifacts/`](../../03_ml_layer_hybrid/artifacts/) or `data/artifacts/`. Skipped when `ENABLE_PRICE_PREDICTION=false`.


In [ ]:
from __future__ import annotations

import sys

_ML_HYBRID_DIR = REPO_ROOT / "notebooks" / "03_ml_layer_hybrid"
if str(_ML_HYBRID_DIR) not in sys.path:
    sys.path.insert(0, str(_ML_HYBRID_DIR))

hybrid_bundle: dict | None = None

if ENABLE_PRICE_PREDICTION:
    from yc_hybrid_inference import load_bundle

    try:
        cleanup_memory()
        mem("before hybrid joblib bundle")
        if HYBRID_BUNDLE_PATH is not None and HYBRID_BUNDLE_PATH.exists():
            hybrid_bundle = load_bundle(HYBRID_BUNDLE_PATH)
        else:
            hybrid_bundle = load_bundle()
        print("Hybrid cluster bundle loaded.")
        mem("after hybrid bundle load")
    except Exception as e:
        print(f"  [hybrid] Bundle not loaded (prediction disabled): {type(e).__name__}: {e}")
        hybrid_bundle = None
else:
    print("ENABLE_PRICE_PREDICTION=false — skipping hybrid bundle load.")



### Price prediction helpers

If the query looks like a **model price estimate** and the hybrid bundle is loaded, Gemma extracts the eight fields required by `predict_from_user_input`. Otherwise `prediction_result` stays empty.


In [ ]:
from __future__ import annotations

    import json
    import re

    import ollama
    from typing import Any

    if ENABLE_PRICE_PREDICTION:
        from yc_hybrid_inference import USER_INPUT_KEYS, predict_from_user_input
    else:
        USER_INPUT_KEYS = ()
        predict_from_user_input = None  # type: ignore[misc, assignment]


    def wants_price_prediction(query: str) -> bool:
        # Heuristic: user wants a numeric model estimate (not just fairness wording).
        ql = query.lower()
        patterns = (
            "predict the price",
            "predict resale",
            "price prediction",
            "estimate the resale",
            "estimate resale price",
            "estimated resale",
            "model estimate",
            "what would it sell for",
            "how much would it sell",
            "valuation for this flat",
        )
        if any(p in ql for p in patterns):
            return True
        if "estimate" in ql and "price" in ql:
            return True
        if ("predict" in ql or "prediction" in ql) and ("price" in ql or "resale" in ql):
            return True
        return False


    def extract_property_fields_from_query(query: str, model: str = OLLAMA_MODEL) -> dict[str, Any] | None:
        # Return dict with USER_INPUT_KEYS or None if JSON parse fails.
        if not USER_INPUT_KEYS:
            return None
        _set_ollama_host(OLLAMA_BASE_URL)
        keys = ", ".join(USER_INPUT_KEYS)
        prompt = (
            "Extract Singapore HDB listing fields from this user message for a price model.
"
            "Return ONLY a JSON object with these keys (all required, use best guess if partial):
"
            f"{keys}

"
            "Rules:
"
            "- town: HDB town in ALL CAPS e.g. TAMPINES
"
            "- flat_type: one of 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE
"
            "- floor_area_sqm: number
"
            '- storey_range: e.g. "07 TO 09" or "10 TO 12"
'
            "- lease_commence_date: integer year
"
            '- sale_month: string "YYYY-MM" for valuation month
'
            "- block, street_name: as in the message

"
            "If you cannot fill a field, use null and we will skip prediction.

"
            f"User message:
{query}
"
        )
        try:
            response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
            raw = (response.get("message") or {}).get("content", "{}")
            raw = re.sub(r"```[\w]*", "", raw).strip()
            parsed = json.loads(raw)
            return parsed if isinstance(parsed, dict) else None
        except Exception as e:
            print(f"  [hybrid] Field extraction failed: {e}")
            return None


    def _format_prediction_block(out: dict[str, Any]) -> str:
        price = out.get("predicted_resale_price")
        lines = [
            f"Predicted resale price (hybrid cluster model): S$ {float(price):,.0f}",
            f"Feature table row matched: {out.get('lookup_matched')}",
        ]
        mk = out.get("matched_address_key")
        if mk:
            lines.append(f"Matched address_key: {mk}")
        note = (out.get("imputation_note") or "").strip()
        if note:
            lines.append(f"Note: {note}")
        return "
".join(lines)


    def compute_prediction_result(query: str) -> str:
        # Non-empty only when bundle loaded, intent matches, fields complete.
        if not ENABLE_PRICE_PREDICTION or hybrid_bundle is None or predict_from_user_input is None:
            return ""
        if not wants_price_prediction(query):
            return ""
        raw_fields = extract_property_fields_from_query(query)
        if not raw_fields:
            return (
                "Could not extract structured fields for the hybrid model. "
                "Provide block, street, town, flat type, floor area (sqm), storey range, "
                "lease commencement year, and sale month (YYYY-MM)."
            )
        missing = [k for k in USER_INPUT_KEYS if raw_fields.get(k) in (None, "", [])]
        if missing:
            return (
                f"Missing or empty fields for hybrid prediction: {missing}. "
                "Add those details to your question."
            )
        try:
            fa = float(raw_fields["floor_area_sqm"])
            lc = int(raw_fields["lease_commence_date"])
        except (TypeError, ValueError) as e:
            return f"Invalid floor_area_sqm or lease_commence_date: {e}"

        kwargs: dict[str, Any] = {
            "block": str(raw_fields["block"]).strip(),
            "street_name": str(raw_fields["street_name"]).strip(),
            "town": str(raw_fields["town"]).strip(),
            "flat_type": str(raw_fields["flat_type"]).strip(),
            "floor_area_sqm": fa,
            "storey_range": str(raw_fields["storey_range"]).strip(),
            "lease_commence_date": lc,
            "sale_month": str(raw_fields["sale_month"]).strip(),
            "bundle": hybrid_bundle,
        }
        if YC_CSV_PATH is not None and YC_CSV_PATH.exists():
            kwargs["yc_csv"] = YC_CSV_PATH

        try:
            out = predict_from_user_input(**kwargs)
            return _format_prediction_block(out)
        except Exception as e:
            return f"Hybrid prediction error: {type(e).__name__}: {e}"


    print("Price prediction helpers defined.")


### NLP filter extraction + namespace routing

Lightweight pre-processing before retrieval:

- **Filter extraction:** Gemma extracts `town`, `flat_type`, `sale_year` from the query → Pinecone metadata filter on the transactions namespace only.
- **Namespace routing:** keyword-based selection of which namespaces to query. Always includes transactions.


In [ ]:
from __future__ import annotations
import json, re
import ollama


def _set_ollama_host(base_url: str) -> None:
    os.environ["OLLAMA_HOST"] = base_url


def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """Use Gemma 3 to extract Pinecone metadata filters from a free-text query."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": ALL CAPS HDB town e.g. "TAMPINES", "BEDOK", "SERANGOON"
  - "flat_type": one of "2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"
  - "sale_year": integer year if mentioned
Omit any field you are not sure about. Return {{}} if nothing is clear.
Return ONLY JSON, no explanation.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if parsed else None
    except Exception as e:
        print(f"  Filter extraction failed: {e}")
        return None


def route_namespaces(query: str) -> list[str]:
    """Select Pinecone namespaces to query based on keywords."""
    q  = query.lower()
    ns = [NS_TRANSACTIONS]
    if any(kw in q for kw in ["mrt", "school", "mall", "hawker", "near", "amenity", "transport", "bus"]):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in ["trend", "rising", "falling", "increase", "decrease", "history",
                               "recent", "last year", "past", "over time", "appreciation"]):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in ["explain", "shap", "feature", "why", "reason", "driver",
                               "factor", "importan", "predict", "model say"]):
        ns.append(NS_XAI)
    return ns


# Smoke test
test_q = "Is $580k fair for a 4-room in Tampines?"
print(f"Query      : {test_q}")
print(f"Filters    : {extract_filters_from_query(test_q)}")
print(f"Namespaces : {route_namespaces(test_q)}")


Query      : Is $580k fair for a 4-room in Tampines?
Filters    : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
Namespaces : ['transactions']


### Hybrid retrieval + source-weighted RRF


In [ ]:
from __future__ import annotations
from typing import Any, Optional
import numpy as np


def _scale_sparse(sparse: dict, scale: float) -> dict:
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    index,
    query: str,
    alpha: float,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict[str, Any]]:
    """Single Pinecone hybrid query. alpha=1.0 → pure dense, 0.0 → pure sparse."""
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))
    res    = index.query(
        vector=dense, sparse_vector=sparse, top_k=int(top_k),
        namespace=namespace, include_metadata=True, filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id": getattr(m, "id", m.get("id")),
         "score": getattr(m, "score", m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


def retrieve_from_namespace(
    index,
    query: str,
    namespace: str,
    top_k: int,
    metadata_filter: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """Dense + sparse retrieval from one namespace."""
    dense  = _hybrid_query(index, query, alpha=1.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    sparse = _hybrid_query(index, query, alpha=0.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    return dense, sparse


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
    source_weights: dict[str, float] | None = None,
) -> list[dict[str, Any]]:
    """
    Merge ranked lists using RRF with source-aware weights.

    score(d) = Σ  weight(source) × 1 / (k + rank_i(d))
    """
    weights = source_weights or SOURCE_WEIGHTS
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}
    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid = str(r.get("id", ""))
            if not rid:
                continue
            source = str((r.get("metadata") or {}).get("source", "transaction"))
            weight = weights.get(source, 1.0)
            scores[rid] = scores.get(rid, 0.0) + weight * (1.0 / (float(k) + float(rank)))
            if rid not in best:
                best[rid] = r
    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


print("Retrieval functions defined (source-weighted RRF active).")


Retrieval functions defined (source-weighted RRF active).


### Reranking funnel

Cross-encoder → MMR → lost-in-middle reorder. 50 → 10 → 5.


In [ ]:
from __future__ import annotations


def _get_text(candidate: dict, field: str = "parent_text") -> str:
    md = candidate.get("metadata") or {}
    return str(md.get(field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
    device: str = CROSS_ENCODER_DEVICE,
) -> list[dict[str, Any]]:
    """Score (query, passage) pairs with the cross-encoder; return top_k."""
    if not candidates:
        return []
    pairs  = [(query, _get_text(c)) for c in candidates]
    inputs = tokenizer(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 1e-12 else 0.0


def mmr_filter(
    candidates: list[dict[str, Any]],
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
) -> list[dict[str, Any]]:
    """Select top_k diverse candidates via Maximal Marginal Relevance."""
    if not candidates:
        return []
    texts   = [_get_text(c) for c in candidates]
    doc_emb = np.asarray(
        dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32
    )
    selected:  list[int] = []
    remaining: list[int] = list(range(len(candidates)))
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)
    while remaining and len(selected) < int(top_k):
        best_idx, best_val = None, -1e18
        for i in remaining:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score   = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val, best_idx = score, i
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [candidates[i] for i in selected]


def reorder_for_context_window(
    candidates: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Lost-in-the-middle mitigation: best first, second-best last."""
    if len(candidates) <= 2:
        return list(candidates)
    ordered = sorted(
        candidates,
        key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)),
        reverse=True,
    )
    return [ordered[0], *ordered[2:], ordered[1]]


print("Reranking funnel defined.")


Reranking funnel defined.


### Multi-query retrieval

Generate N sub-queries via Gemma, fan out across namespaces, fuse with weighted RRF.


In [ ]:
from __future__ import annotations


def generate_subqueries(
    query: str,
    n: int = N_SUBQUERIES,
    model: str = OLLAMA_MODEL,
) -> list[str]:
    """Generate n reformulations of the query using Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""You are a search query generator for Singapore HDB property data.
Generate {n} alternative search queries to help retrieve relevant data from a vector
database of HDB transactions, amenities, price trends, and SHAP features.
Return ONLY a numbered list. No explanations.

Original: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "")
        lines    = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception as e:
        print(f"  Sub-query generation failed: {e}")
        return []


def multi_query_retrieve(
    query: str,
    index,
    namespaces: list[str],
    top_k: int,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """Multi-query hybrid retrieval across all routed namespaces with weighted RRF."""
    all_queries = [query] + generate_subqueries(query)
    all_lists: list[list[dict]] = []
    for q in all_queries:
        for ns in namespaces:
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            dense, sparse = retrieve_from_namespace(index, q, ns, top_k, filt)
            all_lists.extend([dense, sparse])
    return reciprocal_rank_fusion(all_lists, k=RRF_K)


print("Multi-query retrieval defined.")


Multi-query retrieval defined.


### Full retrieval pipeline + defensive smoke test

Smoke test wrapped in try/except; `mem()` logged at each stage so any future crash tells you exactly which stage failed.


In [ ]:
from __future__ import annotations


def retrieve_and_rerank(
    query: str,
    index,
    verbose: bool = False,
) -> list[dict]:
    """Full RAG retrieval pipeline for a free-text query."""
    if verbose: mem("  retr: start")
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)
    if verbose: mem("  retr: after filter+route")

    fused = multi_query_retrieve(
        query=query, index=index, namespaces=namespaces,
        top_k=TOP_K_RETRIEVAL, metadata_filter=metadata_filter,
    )
    if verbose: mem(f"  retr: after fusion ({len(fused)} fused)")

    reranked = rerank_cross_encoder(
        query=query, candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer, model=ce_model, top_k=TOP_K_RERANK,
    )
    if verbose: mem(f"  retr: after CE rerank ({len(reranked)} ranked)")

    diverse = mmr_filter(
        candidates=reranked, query=query,
        top_k=TOP_K_MMR, lambda_param=MMR_LAMBDA,
    )
    if verbose: mem(f"  retr: after MMR ({len(diverse)} diverse)")

    final = reorder_for_context_window(diverse)[:TOP_K_FINAL]
    if verbose: mem("  retr: after reorder")
    return final


# ── Defensive smoke test ────────────────────────────────────────────────────
cleanup_memory()
mem("before retrieval smoke test")
print("Smoke test (verbose mem tracking):")
try:
    smoke_ctx = retrieve_and_rerank(
        "Is $580k fair for a 4-room flat in Tampines?",
        index,
        verbose=True,
    )
    print(f"\n✓ Smoke test passed: {len(smoke_ctx)} chunks retrieved")
    for i, c in enumerate(smoke_ctx, 1):
        m = c.get("metadata") or {}
        print(f"  [{i}] {m.get('source')} | {m.get('town')} | "
              f"{m.get('flat_type','')} | {m.get('sale_year','')}")
except Exception as e:
    print(f"\n✗ Smoke test failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

mem("after smoke test")


Smoke test (verbose mem tracking):
  [MEM]   retr: start                    RSS =   2362.0 MB
  [MEM]   retr: after filter+route       RSS =   2362.0 MB
  [MEM]   retr: after fusion (59 fused)  RSS =   2364.4 MB
  [MEM]   retr: after CE rerank (10 ranked) RSS =   3739.4 MB
  [MEM]   retr: after MMR (5 diverse)    RSS =   3742.1 MB
  [MEM]   retr: after reorder            RSS =   3742.1 MB

✓ Smoke test passed: 5 chunks retrieved
  [1] transaction | TAMPINES | 4 ROOM | 2025
  [2] transaction | TAMPINES | 4 ROOM | 2022
  [3] transaction | TAMPINES | 4 ROOM | 2017
  [4] transaction | TAMPINES | 4 ROOM | 2021
  [5] transaction | TAMPINES | 4 ROOM | 2017
  [MEM] after smoke test                 RSS =   3742.1 MB


### Prompt builder + `generate_answer`

When `compute_prediction_result` returns a non-empty string, it is passed as `prediction_result` and appears under **## Model prediction** in the user prompt. Otherwise that section is omitted.


In [ ]:
from __future__ import annotations

from typing import Any

def build_system_prompt() -> str:
    """System prompt for Gemma 3."""
    return """You are a Singapore HDB property pricing assistant for PropertyLens.
Help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context. No outside knowledge.
2. Cite every specific claim with [Context N] labels.
3. If evidence is thin or contradictory, say so clearly.
4. Keep answers to 3-5 sentences unless detail is requested.
5. Give a Fair / Above market / Below market verdict ONLY for price fairness questions.
   For amenity, trend, or explanation questions, do NOT give a price verdict.
"""


def build_rag_prompt(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
) -> str:
    """Build the user-turn prompt with labelled context chunks."""
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md_  = c.get("metadata") or {}
        txt  = str(md_.get("parent_text") or "").strip()
        hdr  = f"[Context {i}] source={md_.get('source')} town={md_.get('town')} year={md_.get('sale_year')}"
        parts.extend([hdr, txt, ""])
    if prediction_result:
        parts.extend(["## Model prediction", prediction_result, ""])
    parts.extend(["## Question", query])
    return "\n".join(parts).strip()


def generate_answer(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
    model: str = OLLAMA_MODEL,
) -> str:
    """Grounded RAG answer using Ollama + Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user",   "content": build_rag_prompt(query, context_chunks, prediction_result)},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


print("Prompt builder and generate_answer defined.")


Prompt builder and generate_answer defined.


### End-to-end demo

Same demo queries as Notebook B, plus one **hybrid price estimate** query when the bundle is loaded. Each run calls `compute_prediction_result` (may be empty) then `generate_answer` with RAG context.


In [ ]:
from __future__ import annotations

DEMO_QUERIES = [
    {"query": "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
     "persona": "Buyer"},
    {"query": "What should I list my 5-room Bishan flat for given current market trends?",
     "persona": "Seller"},
    {"query": "Are HDB prices in Queenstown rising or falling over the last 3 years?",
     "persona": "Trends"},
    {"query": "What amenities are near Bedok North? Any MRT stations or schools?",
     "persona": "Amenities"},
    {"query": "Why did the model predict a high price for this Queenstown flat? What features drove it?",
     "persona": "XAI"},
    {"query": "Is $430,000 fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon? "
              "64 sqm, 52 years lease remaining, lease started 1978, "
              "3 mins walk to Serangoon MRT.",
     "persona": "PropertyGuru listing"},
    {"query": "Should I buy a 4-room flat in Tampines or Bedok? "
              "Compare prices, trends, and nearby amenities.",
     "persona": "Cross-source comparison"},
    {"query": "The seller is asking $650k for a 5-room in Sengkang. "
              "What is a reasonable counter-offer based on recent sales?",
     "persona": "Negotiation"},
    {"query": "Estimate resale price for Block 432 TAMPINES STREET 81, town TAMPINES, 4 ROOM, floor area 92 sqm, storey range 07 TO 09, lease commence 1990, sale month 2025-03.",
     "persona": "Hybrid price estimate"},
]


def _print_chunk(i: int, c: dict) -> None:
    m      = c.get("metadata") or {}
    source = m.get("source", "")
    if source == "transaction":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] transaction | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")
    elif source == "amenity":
        print(f"    [{i}] amenity | {m.get('town')} | {m.get('amenity_type')} | count={m.get('count')}")
    elif source == "trend":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] trend | {m.get('town')} | median={rp_str} | {m.get('sale_year')}")
    elif source == "xai":
        preview = str(m.get("parent_text", ""))[:80]
        print(f"    [{i}] xai | type={m.get('xai_type')} | {preview}...")
    else:
        print(f"    [{i}] {source} | {m}")


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end. Isolated so failures don't cascade."""
    query = demo["query"]
    print(f"\n{'='*60}")
    print(f"[{demo['persona']}]")
    print(f"{query}")
    print(f"{'='*60}")
    try:
        metadata_filter = extract_filters_from_query(query)
        namespaces      = route_namespaces(query)
        print(f"  Filter     : {metadata_filter}")
        print(f"  Namespaces : {namespaces}")

        mem("before retrieve")
        ctx = retrieve_and_rerank(query, index)
        mem("after retrieve")
        print(f"\n  Context chunks ({len(ctx)}):")
        for i, c in enumerate(ctx, 1):
            _print_chunk(i, c)

        mem("before prediction")
        pred = compute_prediction_result(query)
        mem("after prediction")
        if pred:
            print(f"\n  Model prediction (for prompt):\n{pred}\n")

        mem("before generate_answer")
        answer = generate_answer(query, ctx, prediction_result=pred)
        mem("after generate_answer")
        print(f"\n  Answer:\n{answer}")
    except Exception as e:
        print(f"\n  ✗ Demo failed: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()


cleanup_memory()
mem("before end-to-end demo loop")

for demo in DEMO_QUERIES:
    run_demo(demo)

mem("after demos")



[Buyer]
Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?
  Filter     : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
  Namespaces : ['transactions']
  [MEM] before retrieve                  RSS =   3742.2 MB
  [MEM] after retrieve                   RSS =   3923.4 MB

  Context chunks (5):
    [1] transaction | TAMPINES | 4 ROOM | $558,000 | 2025
    [2] transaction | TAMPINES | 4 ROOM | $439,800 | 2017
    [3] transaction | TAMPINES | 4 ROOM | $513,000 | 2017
    [4] transaction | TAMPINES | 4 ROOM | $410,000 | 2017
    [5] transaction | TAMPINES | 4 ROOM | $488,000 | 2017
  [MEM] before generate_answer           RSS =   3923.4 MB
  [MEM] after generate_answer            RSS =   3923.4 MB

  Answer:
Based on the available data, a 4-room HDB in Tampines Street 81, Block 432 would likely command a price closer to the 2025 transaction data [Context 1]. That transaction shows a 4-room flat with an area of 84.0 sqm selling for SGD 558,000 with an approximate PSF of SGD 617.1

### Notes

- **Hybrid prediction:** Optional `hybrid_cluster_bundle.joblib` + `predict_from_user_input` when `ENABLE_PRICE_PREDICTION=true`. Set `ENABLE_PRICE_PREDICTION=false` to match Notebook B memory use. For lean RAG-only demos, use [`05_propertylens_rag_inference (1).ipynb`](05_propertylens_rag_inference%20(1).ipynb).
- **Restarting the kernel is cheap:** no CSVs, no chunk building. Re-running takes longer when the hybrid bundle loads (~+1 GB RSS).
- **`mem()` checkpoints** around retrieve, prediction, and generate_answer show where memory spikes.
- **If RSS climbs past ~8 GB on a 16 GB Mac during demos:** try `ollama stop gemma3` between sessions, `ENABLE_PRICE_PREDICTION=false`, or reduce `TOP_K_RERANK`.
- **BM25 cache is required** — if `bm25_encoder_v3.pkl` is missing, this notebook fails at the load-encoders cell.
